# Supplemental: Pandas Fundamentals

How to "read" this notebook
<details>
<summary>click to expand</summary>

This supplemental notebook is a hands-on primer for working with **pandas** — Python's core library for tabular data. It covers the core skills needed for data loading, transformation, aggregation, and feature engineering in Python.
<br />

### Learn by doing
Read the explanations and run every code cell. Many cells invite you to experiment — change a filter condition, add a new column, try a different aggregation. You will learn much faster by breaking things and fixing them than by passively reading.
<br />

### Make this notebook your own
Add your own notes, test new ideas, and write down questions as they arise. At the end of the notebook there is a reflection section — use it.
</details>

## Learning Objectives

By the end of this supplement, you will be able to:
- Load a dataset and quickly understand its shape, types, and distributions
- Select columns and filter rows using boolean conditions and `.loc`/`.iloc`
- Create and transform columns using vectorized operations, `.apply()`, and `.map()`
- Summarize data with `.groupby()`, `.agg()`, `.transform()`, and `pd.pivot_table()`
- Handle missing values and combine DataFrames with `pd.concat()` and `pd.merge()`
- Use the `.str` and `.dt` accessors for string and datetime feature engineering

<div class="callout" style="
  width: 80%;
  background: rgba(127,127,127,0.15);
  border: 1px solid rgba(127,127,127,0.3);
  padding: 10px 30px;
  margin: 20px;
  border-radius: 6px;
  text-align: justify;
  text-align-last: left;
  font-size: 11pt;
">
  <span style="
    font-size: 60pt;
    line-height: 1;
    float: left;
    margin: 0px 0px 0px 0;
  ">
    🧩
  </span>

  This notebook covers the core pandas skills every data scientist needs. If you have used pandas before, skim quickly and spend time on the exercises. If pandas is new to you, work through each section carefully — these are the building blocks for everything that follows.
  <!-- clearfix -->
  <div style="clear: both;"></div>
</div>

# P.0 Code Preface

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Throughout this notebook we will work with **Bluebikes** — Boston's public bike-share system. The dataset contains a sample of individual trips, with information about trip duration, start/end stations, and rider demographics. It is a great dataset for learning pandas because it has numeric columns, categorical columns, string columns, and datetime columns all in one place.

The raw column names include spaces (e.g., `start station name`), which prevents dot-notation access. The first thing we will do after loading is rename all columns to `snake_case`.

In [ ]:
# Load the Bluebikes sample dataset
df = pd.read_csv('../data/bluebikes_sample.csv')

# Rename all columns to snake_case so we can use dot notation throughout
df.columns = df.columns.str.lower().str.replace(' ', '_')

print(f"Loaded {len(df):,} rows and {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")

# P.1 Loading Data and First Look

<img alt="Luke dreaming of what lies ahead" src="../images/LP_luke_vista.png" width="900" style="display:block;">
<font size=2>Luke dreams about what lies ahead in <i>Star Wars (1977)</i></font>

The first thing you should always do with a new dataset is **understand what you have**. How many rows and columns? What types are the columns? Are there missing values? What does a typical row look like?

Pandas gives you several tools to answer these questions quickly.

## Shape, Types, and a Quick Preview

Start with the basics: how big is the DataFrame and what does it contain?

In [ ]:
# .head() shows the first 5 rows (pass a number to change how many)
df.head()

We can also see the shape of the dataframe (how many rows,columns it has), and the data types of each column:

In [ ]:
# .shape is a tuple: (rows, columns)
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

# .dtypes shows the data type of every column
print("\nColumn types:")
print(df.dtypes)

## `.info()` and `.describe()`: Two Essential Diagnostic Tools

Probably the two most important methods for quickly understanding a DataFrame beyond `.head()` are the `.info()` and `.describe()` methods. First, `.info()` gives you a concise summary of the DataFrame — column names, non-null counts, and dtypes — all at once. It's the fastest way to spot missing values:

In [ ]:
# .info() shows column names, non-null counts, and dtypes
# Spot columns with missing values by looking for non-null count < total rows
df.info()

Next, `.describe()` computes summary statistics for all numeric columns, or for a specific column. For categorical columns, pass `include='all'` to include them too.

In [ ]:
# .describe() computes count, mean, std, min, quartiles, max for numeric columns
df.describe()

In [ ]:
# Describe a specific column:
df.tripduration.describe()

## `.value_counts()`: Frequency Tables for Categorical Columns

For categorical (non-numeric) columns, `.value_counts()` is more useful than `.describe()`. It shows how often each unique value appears.

In [ ]:
# How many trips by each user type?
print(df.usertype.value_counts())

Sometimes you want to know the percentage breakdown instead of raw counts. If so, you can add `normalize=True` to get proportions:

In [ ]:
# You can add normalize=True gives proportions instead of counts
print(df.usertype.value_counts(normalize=True).round(3))

<!-- Start Exercise P.1 -->
<hr/>
<img src="../images/stop_right_margin.png" align="left">

<font size=3 color="darkred"> Exercise: Profile the Dataset </font>
<div class="inclass_exercise_body" style="padding-left: 130px; width: 85%; text-align: justify;text-align-last: left;">
Using the tools from P.1, answer the following questions about the Bluebikes dataset:
<ol>
<li>How many trips have a missing <code>birth_year</code>? (Hint: use <code>.info()</code> or <code>.isnull().sum()</code>)</li>
<li>What is the mean and maximum <code>tripduration</code>?</li>
<li>What are the top 5 most common <code>start_station_name</code> values?</li>
</ol>
</div>

In [ ]:
# 1. How many trips are missing birth_year?
# Your code here

# 2. Mean and max tripduration
# Your code here

# 3. Top 5 most common start stations
# Your code here

<hr/>
<!-- End Exercise P.1 -->

# P.2 Selecting, Filtering, and Slicing

<img alt="Gandalf stop the Balrog" src="../images/LP_gandalf_balrog.png" width="900" style="display:block;">
<font size=2>"You shall not pass!"- Gandalf filters the Balrog in <i>The Lord of the Rings:The Fellowship of the Ring (2001)</i></font>

Most real-world analysis starts by pulling out the rows and columns you actually need. Pandas gives you several ways to do this.

## Selecting Columns

Use dot notation (or bracket notation) to access a single column as a **Series**:

In [ ]:
# Single column → Series (note: a Series is like a one-column DataFrame)
durations = df.tripduration # I personally prefer this dot notation style when possible
# durations = df['tripduration']  # equivalent using bracket notation; depending on column names, you may need to use bracket notation (e.g. df['start station name'])
print(type(durations))   # pandas Series
print(durations.head())

## Filtering Rows with Boolean Conditions

A **boolean filter** is a Series of `True`/`False` values the same length as the DataFrame. Passing it inside `df[...]` keeps only the `True` rows.

In [ ]:
# Keep only Subscriber trips
subscribers = df[df.usertype == 'Subscriber']
print(f"Subscriber trips: {len(subscribers):,}")

# Keep only long trips (over 30 minutes = 1800 seconds)
long_trips = df[df.tripduration > 1800]
print(f"Trips over 30 min: {len(long_trips):,}")

# Combine conditions with & (and) and | (or) — always wrap conditions in parentheses!
long_subscriber_trips = df[(df.usertype == 'Subscriber') & (df.tripduration > 1800)]
print(f"Long Subscriber trips: {len(long_subscriber_trips):,}")

## `.loc[]` and `.iloc[]`: Precise Row and Column Selection

- **`.loc[]`** selects by **label** (row index value or column name)
- **`.iloc[]`** selects by **integer position** (0-based)

Both accept `[rows, columns]` syntax. Use **`:`** in either position to mean *"everything"* — just like Python slice notation. `df.loc[:, 'col']` means all rows, one column. `df.loc[condition, :]` means filtered rows, all columns.

In [ ]:
# : in the row position means "all rows" — select one column across every row
df.loc[:, 'tripduration']

In [ ]:
# : in the column position means "all columns" — equivalent to just df[condition]
df.loc[df.usertype == 'Subscriber', :].head() # we added .head() just to limit the output for display purposes

In [ ]:
# Both positions specified: filtered rows AND selected columns
df.loc[df.usertype == 'Subscriber', ['start_station_name', 'tripduration']].head()

In [ ]:
# .iloc uses integer positions; : works the same way
df.iloc[0:3,:] # first 3 rows, all columns

# Other examples:
# df.iloc[0:5, :]        # first 5 rows, all columns
# df.iloc[:, 0:3]        # all rows, first 3 columns
# df.iloc[0:5, 0:3]      # first 5 rows, first 3 columns

## `.query()`: A Readable Filter Shorthand

`.query()` lets you write filter conditions as a readable string — useful for complex filters or when you want the code to read more like English.

In [ ]:
# Equivalent to df.loc[(df.usertype == 'Subscriber') & (df.tripduration > 1800), :]
df.query("usertype == 'Subscriber' and tripduration > 1800").head() # Again, adding .head() just to limit the output for display purposes

If you need to reference Python variables inside a query string, use `@` to access them, like this:

In [ ]:
# You can reference Python variables inside a query string with @
min_duration = 600
df.query("tripduration >= @min_duration").head()

<!-- Start Exercise P.2 -->
<hr/>
<img src="../images/stop_right_margin.png" align="left">

<font size=3 color="darkred"> Exercise: Filter and Select </font>
<div class="inclass_exercise_body" style="padding-left: 130px; width: 85%; text-align: justify;text-align-last: left;">
Using what you learned in P.2:
<ol>
<li>Filter the DataFrame to keep only <b>Customer</b> trips that lasted more than 10 minutes (600 seconds).</li>
<li>From that filtered result, select only <code>start_station_name</code>, <code>end_station_name</code>, and <code>tripduration</code>.</li>
<li>How many such trips are there? What is the average duration?</li>
</ol>
</div>

In [ ]:
# 1 & 2. Filter Customers with trips > 10 minutes, keep selected columns
# Your code here

# 3. Count and average duration
# Your code here

<hr/>
<!-- End Exercise P.2 -->

# P.3 Creating and Transforming Columns

<img alt="MJ turns wolf in Thriller" src="../images/LP_mj_thriller.png" width="600" style="display:block;">
<font size=2>Michael Jackson undergoes a transformation in <i>Thriller (1983)</i></font>

Raw data is rarely in the form you need. You will almost always need to derive new columns — converting units, computing differences, recoding categories, or combining existing fields.

## Vectorized Operations: The Pandas Way

Pandas operations work on entire columns at once — no loops needed. Assigning the result to a new column name creates that column in the DataFrame.

In [ ]:
# Create a new column trip_minutes from tripduration
df['trip_minutes'] = df.tripduration / 60  

Note that, when we are creating a new column, we have to use the bracket notation `df['new_col']` instead of dot notation. This is because dot notation only works for existing columns.

In [ ]:
# Create a new column rider_age from birth_year (assuming 2020 as the approximate year of the data)
df['rider_age'] = 2020 - df.birth_year

Now we can use aggregation operators on the columns we created like this:

In [ ]:
print(f"Trip minutes - min: {df.trip_minutes.min():.1f}, max: {df.trip_minutes.max():.1f}")
print(f"Rider age - mean: {df.rider_age.mean():.1f}, range: {df.rider_age.min():.0f}–{df.rider_age.max():.0f}")

## `.apply()` and Lambda: Row-by-Row Transformations

Vectorized operations are always preferrable because they are more efficient. When a vectorized expression isn't enough, `.apply()` runs a function on every element (or every row). This works well in combination with lambda functions.  **lambda** is an inline anonymous function — useful for short one-liners.

In [ ]:
# Classify trips into duration buckets using apply + lambda
def classify_trip(minutes):
    if minutes < 10:
        return 'short'
    elif minutes < 30:
        return 'medium'
    else:
        return 'long'

df['trip_length_cat'] = df.trip_minutes.apply(classify_trip)

# Or equivalently with a lambda (for simpler logic)
df['trip_length_cat'] = df.trip_minutes.apply(
    lambda x: 'short' if x < 10 else ('medium' if x < 30 else 'long')
)

print(df.trip_length_cat.value_counts())

## `.map()`: Recode a Column with a Dictionary

`.map()` is perfect for replacing values using a lookup dictionary. Much cleaner than a chain of `.replace()` calls.

In [ ]:
# The gender column uses numeric codes: 0=unknown, 1=male, 2=female
gender_map = {0: 'unknown', 1: 'male', 2: 'female'}
df['gender_label'] = df.gender.map(gender_map)

print(df.gender_label.value_counts())

<div class="callout" style="
  width: 80%;
  background: rgba(127,127,127,0.15);
  border: 1px solid rgba(127,127,127,0.3);
  padding: 10px 30px;
  margin: 20px;
  border-radius: 6px;
  text-align: justify;
  text-align-last: left;
  font-size: 11pt;
">
  <span style="
    font-size: 60pt;
    line-height: 1;
    float: left;
    margin: 0px 0px 0px 0;
  ">
    💡
  </span>

  Prefer **vectorized operations** (arithmetic, comparisons) over `.apply()` whenever possible — they are much faster on large datasets. Use `.apply()` when the logic is genuinely complex. Use `.map()` specifically for value recoding via a dictionary.
  <!-- clearfix -->
  <div style="clear: both;"></div>
</div>

<!-- Start Exercise P.3 -->
<hr/>
<img src="../images/stop_right_margin.png" align="left">

<font size=3 color="darkred"> Exercise: Engineer New Features </font>
<div class="inclass_exercise_body" style="padding-left: 130px; width: 85%; text-align: justify;text-align-last: left;">
Add two new columns to the DataFrame:
<ol>
<li><code>is_round_trip</code>: a boolean column that is <code>True</code> when <code>start_station_id == end_station_id</code>.</li>
<li><code>age_group</code>: a string column that categorizes <code>rider_age</code> into <code>'under_30'</code>, <code>'30_to_50'</code>, and <code>'over_50'</code>. Use <code>.apply()</code> with a lambda or a named function.</li>
</ol>
Then print the value counts for each new column.
</div>

In [ ]:
# 1. is_round_trip
# Your code here

# 2. age_group
# Your code here

# Print value counts
# Your code here

<hr/>
<!-- End Exercise P.3 -->

# P.4 Grouping and Aggregating

<img alt="The goonies stick together" src="../images/LP_goonies_group.png" width="900" style="display:block;">
<font size=2>The goonies stick together in <i>The Goonies (1985)</i></font>

The most powerful tool in pandas for summarizing data is **`.groupby()`**. It splits the DataFrame into groups, applies an aggregation function to each group, and combines the results.

This section is the most important one in this supplement — `groupby` is listed as a prerequisite in Lectures 5 and 6 and is used constantly in real-world analytics.

## Basic `.groupby()` + `.agg()`

In [ ]:
# Average trip duration by user type
print(df.groupby('usertype').trip_minutes.mean().round(1))

# Count trips by user type
print("\nTrip counts:")
print(df.groupby('usertype').trip_minutes.count())

## Multiple Aggregations at Once

Pass a dictionary to `.agg()` to compute several statistics for multiple columns in one call.

In [ ]:
# Aggregate several statistics at once
summary = df.groupby('usertype').agg(
    trip_count=('tripduration', 'count'),
    avg_minutes=('trip_minutes', 'mean'),
    median_minutes=('trip_minutes', 'median'),
    max_minutes=('trip_minutes', 'max')
).round(1)

print(summary)

## Grouping by Multiple Columns

In [ ]:
# Average trip duration by user type AND trip length category
by_type_length = df.groupby(['usertype', 'trip_length_cat']).agg(
    count=('tripduration', 'count'),
    avg_minutes=('trip_minutes', 'mean')
).round(1)

print(by_type_length)

## The Index After `.groupby()`: `reset_index()`

When you call `.groupby().agg()`, the group column(s) become the **index** of the result — they are no longer regular columns. This is fine for display, but it causes problems if you want to:
- merge the result back onto another DataFrame
- reference the group column by name with dot notation
- sort or filter by the group column

The fix is `.reset_index()`, which promotes the index back to a regular column:


In [ ]:
# After groupby, the group column becomes the index
summary = df.groupby('usertype').agg(
    trip_count=('tripduration', 'count'),
    avg_minutes=('trip_minutes', 'mean')
).round(1)

print("usertype is now the INDEX, not a column:")
print(summary)
print(f"\n{summary.index=}")
print(f"Columns: {summary.columns.tolist()}")

In [ ]:
# .reset_index() promotes the index back to a regular column
summary_flat = summary.reset_index()

print("After reset_index() — usertype is a regular column again:")
print(summary_flat)
print(f"\n{summary_flat.index=}")
print(f"Columns: {summary_flat.columns.tolist()}")

# Now you can reference the group column with dot notation
print(f"\nUser types: {summary_flat.usertype}")

# Which would allow you to e.g., merge it onto another DataFrame using 'usertype' as a key
# pd.merge(df, summary_flat, on='usertype')  # this would fail without reset_index()

## `.transform()`: Add a Group Statistic Back to Every Row

`.groupby().agg()` returns a *smaller* DataFrame with one row per group. Sometimes you want to add that group statistic back to the *original* DataFrame — for example, to compute how each trip compares to its user-type's average. That's what `.transform()` does.

In [ ]:
# Add each user type's average trip duration as a new column (same length as df)
df['usertype_avg_minutes'] = df.groupby('usertype').trip_minutes.transform('mean')

# Now compute how much each trip deviates from its user-type average
df['minutes_vs_avg'] = df.trip_minutes - df.usertype_avg_minutes

df[['usertype', 'trip_minutes', 'usertype_avg_minutes', 'minutes_vs_avg']].head(8)

## `pd.pivot_table()`: Cross-Tabulated Summaries

A pivot table is a grouped aggregation presented in a matrix format — rows are one grouping variable, columns are another.

In [ ]:
# Average trip minutes cross-tabulated by user type and trip length category
pivot = pd.pivot_table(
    df,
    values='trip_minutes',
    index='usertype',
    columns='trip_length_cat',
    aggfunc='count',
    fill_value=0
)

print(pivot)

<!-- Start Exercise P.4 -->
<hr/>
<img src="../images/stop_right_margin.png" align="left">

<font size=3 color="darkred"> Exercise: Groupby Analysis </font>
<div class="inclass_exercise_body" style="padding-left: 130px; width: 85%; text-align: justify;text-align-last: left;">
Using <code>.groupby()</code> and <code>.agg()</code>:
<ol>
<li>Calculate the <b>mean</b> and <b>median</b> trip duration (in minutes) for each <code>gender_label</code> group.</li>
<li>Find the 5 <code>start_station_name</code> values with the <b>highest average trip duration</b>.</li>
</ol>
Bonus: Build a pivot table showing average <code>trip_minutes</code> with <code>usertype</code> as rows and <code>gender_label</code> as columns.
</div>

In [ ]:
# 1. Mean and median trip minutes by gender_label
# Your code here

# 2. Top 5 start stations by average trip duration
# Your code here

# Bonus: pivot table
# Your code here

<hr/>
<!-- End Exercise P.4 -->

# P.5 Handling Missing Data and Combining DataFrames

<img alt="Chevy Chase in memoirs of the invisible man" src="../images/LP_invisibleman_fillna.png" width="900" style="display:block;">
<font size=2>Chevy Chase tries to <code>fillna</code> in <i>Memoirs of an Invisible Man (1992)</i></font>

Real-world datasets are messy. Missing values are the norm, not the exception. And data is almost never stored in a single table — you will need to combine it from multiple sources.

## Detecting and Handling Missing Values

First let's make a copy of our bluebikes DataFrame and add some simulated missingness:

In [ ]:
# Make a copy of our bluebikes dataframe and add missingness
df_missing = df.copy()

# Randomly set 10% of birth_year values to NaN
np.random.seed(42)  # for reproducibility
missing_mask = np.random.rand(len(df_missing)) < 0.1
df_missing.loc[missing_mask, 'birth_year'] = np.nan

# Randomly set 10% of start_station_name values to NaN
missing_mask = np.random.rand(len(df_missing)) < 0.1
df_missing.loc[missing_mask, 'start_station_name'] = np.nan

To handle missing values, you first need to detect them. Use `.isnull()` to get a boolean mask of missing values, and `.sum()` to count them.

In [ ]:
# Count missing values in each column
print("Missing values per column:")
print(df_missing.isnull().sum())

# As a percentage
print("\nMissing as % of total:")
print((df_missing.isnull().sum() / len(df_missing) * 100).round(1))

To drop rows with missing values, use `.dropna()`:

In [ ]:
# .dropna() removes rows with any missing value
df_no_missing = df_missing.dropna()
print(f"Original rows: {len(df_missing):,}")
print(f"After dropna: {len(df_no_missing):,}")

In [ ]:
# .dropna(subset=...) removes rows missing in specific columns only
df_known_age = df_missing.dropna(subset=['birth_year'])
print(f"Rows with known birth year: {len(df_known_age):,}")

To fill them with a specific value, use `.fillna()`:

In [ ]:
# .fillna() fills missing values
df_missing['birth_year_filled'] = df_missing.birth_year.fillna(df_missing.birth_year.median())
print(f"\nMissing birth_year after fillna: {df_missing.birth_year_filled.isnull().sum()}")

## `pd.concat()`: Stacking DataFrames

Often you will have two dataframes with exactly the same columns but different rows -- for example, data from two different months -- and you want to combine them into one big dataframe. `pd.concat()` stacks DataFrames vertically (row-wise; much more common use-case) or horizontally (column-wise). Most commonly used to combine data from multiple files or time periods. If you are using `pd.concat()` for column-wise concatenation, ask your self if you shouldn't instead be merging on a key column with `pd.merge()`  (discussed next).

In [ ]:
# Split dataset into two halves (simulating data from two files)
half = len(df) // 2
df_part1 = df.iloc[:half].copy()
df_part2 = df.iloc[half:].copy()

print(f"Part 1: {len(df_part1):,} rows")
print(f"Part 2: {len(df_part2):,} rows")

# Stack them back together
df_combined = pd.concat([df_part1, df_part2], ignore_index=True)
print(f"Combined: {len(df_combined):,} rows")

## `pd.merge()`: Joining DataFrames on a Key

`pd.merge()` combines two DataFrames using a shared key column — like a SQL JOIN. The `how` parameter controls which rows to keep:
- `'inner'` — only rows where the key exists in **both** DataFrames (default)
- `'left'` — all rows from the left DataFrame, matched rows from the right
- `'outer'` — all rows from both DataFrames

<div class="callout" style="
  width: 80%;
  background: rgba(127,127,127,0.15);
  border: 1px solid rgba(127,127,127,0.3);
  padding: 10px 30px;
  margin: 20px;
  border-radius: 6px;
  text-align: justify;
  text-align-last: left;
  font-size: 11pt;
">
  <span style="
    font-size: 60pt;
    line-height: 1;
    float: left;
    margin: 0px 0px 0px 0;
  ">
    💡
  </span>

  This section covers merge basics. Advanced topics like using merge **indicators** and **validation** (detecting one-to-many vs many-to-many relationships, auditing unmatched rows) build on these foundations.
  <!-- clearfix -->
  <div style="clear: both;"></div>
</div>

Let's use an example with some fabricated data to illustrate a left join:

In [ ]:
# Build a small lookup table of station IDs → borough labels (fabricated example)
np.random.seed(42)
station_ids = df.start_station_id.unique()[:20]  # grab 20 real station IDs
boroughs = np.random.choice(['Back Bay', 'South End', 'Fenway', 'Cambridge', 'Somerville'], 20)

station_info = pd.DataFrame({
    'start_station_id': station_ids,
    'borough': boroughs
})

station_info.head()

Now let's look at a **left join** to add on the borough information for each station (whenever we have it):

In [ ]:
# Left join: keep all trips, add borough where we have station info
df_with_borough = pd.merge(df, station_info, on='start_station_id', how='left')

print(f"Shape after merge: {df_with_borough.shape=}") 

print("\nSome rows/columns to see the merge results:")
df_with_borough.loc[50:100, ['tripduration','start_station_id','borough']].head()  # Look at some rows/columns to see the merge results

We can see that all trips from `df` are still present, some with `NaN` boroughs where there was no match in `stations`. Merging data is a much deeper topic (which type of join do you want? How do you handle duplicates? Do you need to audit unmatched rows?) but this is the basic idea.

<!-- Start Exercise P.5 -->
<hr/>
<img src="../images/stop_right_margin.png" align="left">

<font size=3 color="darkred"> Exercise: Clean and Combine </font>
<div class="inclass_exercise_body" style="padding-left: 130px; width: 85%; text-align: justify;text-align-last: left;">
<ol>
<li>Create a clean version of the dataset called <code>df_clean</code> that drops rows where <code>birth_year</code> is missing and where <code>gender</code> is 0 (unknown).</li>
<li>How many rows remain? What percentage of the original data did you retain?</li>
<li>Split <code>df_clean</code> into Subscribers and Customers, then <code>pd.concat()</code> them back together. Verify the row count matches.</li>
</ol>
</div>

In [ ]:
# 1. Create df_clean
# Your code here

# 2. Count and percentage
# Your code here

# 3. Split and concat
# Your code here

<hr/>
<!-- End Exercise P.5 -->

# P.6 String and Datetime Accessors

<img alt="Phil struggles with datetime" src="../images/LP_groundhog_time.png" width="900" style="display:block;">
<font size=2>Phil struggles with parsing datetime in <i>Groundhog Day (1993)</i></font>

Two of the most common data types in real-world datasets are **strings** and **datetimes**. Pandas provides the `.str` and `.dt` accessors to work with them efficiently — without writing loops.

## The `.str` Accessor: String Operations on a Column

Any string method you know from Python (`lower()`, `split()`, `replace()`, etc.) works on a pandas column via `.str`.

In [ ]:
# Normalize station names to lowercase
df['start_station_lower'] = df.start_station_name.str.lower()

In [ ]:
# Does the station name contain a street abbreviation like 'St' or 'Ave'?
df['station_is_street'] = df.start_station_name.str.contains(r'\bSt\b|Ave|Blvd', regex=True)
print(f"Trips from street-named stations: {df.station_is_street.sum():,}")

In [ ]:
# Extract the first word of the station name (the cross street or landmark)
df['station_first_word'] = df.start_station_name.str.split(' ').str[0]
print("\nTop 5 station name first words:")
print(df.station_first_word.value_counts().head())

## P.6 String and Datetime Accessors

### The `.dt` Accessor: Datetime Feature Extraction

Before using `.dt`, you must convert a string column to a `datetime64` type using `pd.to_datetime()`. After that, `.dt` gives you access to `.year`, `.month`, `.day`, `.hour`, `.day_of_week`, and much more.

The `starttime` column in this dataset encodes **time-of-day as elapsed minutes:seconds since midnight**, stored as a string in `MM:SS.f` format. For example, `"26:46.4"` means 26 minutes and 46.4 seconds after midnight (i.e., 12:26 AM).

In [ ]:
# starttime is stored as MM:SS.f — minutes:seconds.fraction since midnight
# e.g. "26:46.4" = 26 min 46.4 sec after midnight = 12:26 AM
print(df[['starttime', 'stoptime', 'tripduration']].head(5))

Because this sample doesn't include a calendar date, we'll fabricate one — stored as a plain string, just as you'd receive it in a real CSV. That gives us two string columns to convert: a date string (`trip_date`) and a time-of-day string (`starttime`). Converting both to the right types and combining them is a common real-world task.

In [ ]:
# Don't worry about following this code cell right now it just fabricates a trip_date column as a string, simulating how you'd receive date data in a real CSV.
import numpy as np
rng = np.random.default_rng(seed=42)
days_offset = rng.integers(0, 365, size=len(df))
dates = pd.Timestamp('2019-01-01') + pd.to_timedelta(days_offset, unit='D')
df['trip_date'] = dates.strftime('%Y-%m-%d')   # store as a plain string

# Normalize the starttime by prepending "0:" so pandas reads it as H:MM:SS.f
if (df.starttime.str.count(':')==1).sum()==df.shape[0]:  # if it hasn't been normalized yet, do so
    df.starttime = '0:' + df.starttime

print(df[['trip_date', 'starttime']].head())
print(f"\ntrip_date dtype : {df.trip_date.dtype}")   # object (string)
print(f"starttime dtype : {df.starttime.dtype}")    # object (string)

Before we can leverage datetime functionality in pandas, we need to convert strings to datetime objects. Look at the code below which converts our fabrictaed `trip_date` strings to `datetime` objects and then combines the starting time records with our (fabricated) dates to get a real datetime for the beginning of the trip: 

In [ ]:
# convert the date string "YYYY-MM-DD" to datetime64
df['trip_date_dt'] = pd.to_datetime(df.trip_date)

# convert the time-of-day string "H:MM:SS.f" to timedelta64
starttime_td = pd.to_timedelta(df.starttime)

# combine date + time-of-day into a full datetime column
df['starttime_dt'] = df.trip_date_dt + starttime_td

print(df[['trip_date', 'starttime', 'starttime_dt']].head())
print(f"\nstarttime_dt dtype: {df.starttime_dt.dtype}")

Now we can get some meaningful features from the `datetime`:

In [ ]:
# Now extract .dt features — all are meaningful because we have real dates
df['hour']        = df.starttime_dt.dt.hour         # 0–23
df['day_of_week'] = df.starttime_dt.dt.day_of_week  # 0=Monday … 6=Sunday
df['month']       = df.starttime_dt.dt.month        # 1–12

print("Trip counts by hour of day (top 5):")
print(df.groupby('hour').tripduration.count().sort_values(ascending=False).head())

print("\nTrip counts by day of week (0=Mon, 6=Sun):")
print(df.groupby('day_of_week').tripduration.count())

print("\nTrip counts by month:")
print(df.groupby('month').tripduration.count())

<div class="callout" style="
  width: 80%;
  background: rgba(127,127,127,0.15);
  border: 1px solid rgba(127,127,127,0.3);
  padding: 10px 30px;
  margin: 20px;
  border-radius: 6px;
  text-align: justify;
  text-align-last: left;
  font-size: 11pt;
">
  <span style="
    font-size: 60pt;
    line-height: 1;
    float: left;
    margin: 0px 0px 0px 0;
  ">
    🧠
  </span>

<font color=blue><b>Q</b></font>: Why extract hour, day of week, and month as separate columns instead of keeping the raw datetime?

<font color=blue><b>A</b></font>: Machine learning models treat numbers as continuous values — they can't natively understand that 23:00 and 00:00 are close together, or that the pattern repeats weekly. Extracting components like <code>hour</code> and <code>day_of_week</code> lets the model learn those patterns.
  <!-- clearfix -->
  <div style="clear: both;"></div>
</div>

<!-- Start Exercise P.6 -->
<hr/>
<img src="../images/stop_right_margin.png" align="left">

<font size=3 color="darkred"> Exercise: String and Datetime Features </font>
<div class="inclass_exercise_body" style="padding-left: 130px; width: 85%; text-align: justify;text-align-last: left;">
<ol>
<li>Find the busiest <b>departure hour</b> (by trip count) for each <code>usertype</code>. Do Subscribers and Customers peak at different times?</li>
<li>Create a column <code>same_neighborhood</code> that is <code>True</code> when the first word of <code>start_station_name</code> matches the first word of <code>end_station_name</code>. What percentage of trips are within the same neighborhood?</li>
</ol>
</div>

In [ ]:
# 1. Busiest departure hour by usertype
# Your code here

# 2. same_neighborhood column
# Hint: use str.split(' ').str[0] on both start and end station names
# Your code here

<hr/>
<!-- End Exercise P.6 -->

# Summary

This supplement covered the core pandas skills for data science:

## P.1 Loading Data and First Look
- `pd.read_csv()` to load data; rename columns with `.str.lower().str.replace()` for dot notation
- `.head()`, `.shape`, `.dtypes`, `.info()`, `.describe()`, `.value_counts()` for profiling

## P.2 Selecting, Filtering, and Slicing
- Dot notation (`df.col`) for single columns; `df[['a', 'b']]` for multiple columns
- Boolean filters with `&` and `|`; `.loc[]` for label-based selection; `.query()` for readable filters
- Use `:` in `.loc[]`/`.iloc[]` to mean "all rows" or "all columns"

## P.3 Creating and Transforming Columns
- Vectorized arithmetic for new columns; `.apply()` + lambda for complex row-level logic; `.map()` for dictionary-based recoding

## P.4 Grouping and Aggregating
- `.groupby().agg()` with named aggregations; grouping by multiple keys
- Group keys become the index after `.groupby()` — use `.reset_index()` to promote them back to regular columns
- `.transform()` to broadcast group statistics back to each row; `pd.pivot_table()` for matrix summaries

## P.5 Handling Missing Data and Combining DataFrames
- `.isnull().sum()`, `.dropna(subset=[...])`, `.fillna()` for missing values
- `pd.concat()` to stack DataFrames; `pd.merge()` with `how='left'`/`'inner'`/`'outer'`

## P.6 String and Datetime Accessors
- `.str.lower()`, `.str.contains()`, `.str.split()` for string feature engineering
- `pd.to_datetime()`, `.dt.hour`, `.dt.day_of_week`, `.dt.month` for datetime features

<div class="callout" style="
  width: 80%;
  background: rgba(127,127,127,0.15);
  border: 1px solid rgba(127,127,127,0.3);
  padding: 10px 30px;
  margin: 20px;
  border-radius: 6px;
  text-align: justify;
  text-align-last: left;
  font-size: 11pt;
">
  <span style="
    font-size: 60pt;
    line-height: 1;
    float: left;
    margin: 0px 0px 0px 0;
  ">
    🎯
  </span>
  <div style="padding-left: 80px;">

  **Where to go deeper:**
  - **Efficient file formats**: Parquet, Feather and dtype memory optimization for large files
  - **Time series analysis**: Aggregating and resampling datetime for time series data
  - **Geospatial analysis**: Working with geographic data and spatial operations   
  </div>
  <!-- clearfix -->
  <div style="clear: both;"></div>
</div>

# Your Turn: Questions, Reactions, and Feedback

1. Which concept was most new or surprising to you?

2. Which topic do you want to explore further?

3. Any questions that came up as you worked through the notebook?

4. What connections do you see between these techniques and data work you've done before?

**Write your thoughts below:**

*[Your reflections here]*